In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ModuleNotFoundError: No module named 'transformers'

In [ ]:
model_id = "aisingapore/Apertus-SEA-LION-v4-8B-IT"

model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "Any flats for sale in Tampines?"
    }
]

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_hdb_listings",
            "description": "Search for HDB flats available for sale",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "Town or area name"},
                    "flat_type": {"type": "string", "description": "Flat type e.g. 3-room, 4-room, 5-room"},
                    "max_price": {"type": "number", "description": "Maximum price in SGD"}
                },
                "required": ["location", "flat_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_mortgage",
            "description": "Calculate estimated monthly mortgage payment",
            "parameters": {
                "type": "object",
                "properties": {
                    "loan_amount": {"type": "number", "description": "Loan amount in SGD"},
                    "interest_rate": {"type": "number", "description": "Annual interest rate as percentage"},
                    "loan_tenure_years": {"type": "integer", "description": "Loan period in years"}
                },
                "required": ["loan_amount"]
            }
        }
    }
]

In [ ]:

inputs = tokenizer.apply_chat_template(
    messages, tools=tools, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    generation = generation[0][input_len:]

decoded = tokenizer.decode(generation, skip_special_tokens=True)
print(decoded)
